In [ ]:
!pip install /kaggle/input/rdkit-2025-3-3-cp311/rdkit-2025.3.3-cp311-cp311-manylinux_2_28_x86_64.whl

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler, RobustScaler
from rdkit import Chem
from rdkit.Chem import Descriptors, rdmolops
import warnings
import random
from tqdm import tqdm
import math
import statistics as stat
import pickle
import json
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
# 确保画图的文件夹存在
os.makedirs('./plots', exist_ok=True)

# --- MolFormer 部分被移除 ---
# MolFormer 不再被加载或使用
MOLFORMER_AVAILABLE = False
print("i MolFormer has been intentionally removed for this ablation study.")

warnings.filterwarnings('ignore')

# ===============================
# 配置和超参数
# ===============================
SEED = 42
TARGET_COLUMNS = ['Tg', 'FFV', 'Tc', 'Density', 'Rg']
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 数据路径
BASE_PATH = '/kaggle/input/neurips-open-polymer-prediction-2025/'

# 保持原有模型参数
SMILES_EMBEDDING_DIM = 256
DROPOUT_RATE = 0.05
LEARNING_RATE = 0.001
BATCH_SIZE = 16
EPOCHS = 200
N_FOLDS = 5

# 数据增强配置
DATA_AUGMENTATION = True
AUGMENTATION_PROBABILITY = 0.3

# 目标变量缩放因子
TARGET_SCALING = {
    'FFV': 1000,
    'Tc': 100,
    'Tg': 1,
    'Density': 1,
    'Rg': 1
}

# 保持原有CNN配置
CNN_CONFIGS = [
    {'out_channels': 64, 'kernel_size': 3, 'dilation': 1},
    {'out_channels': 64, 'kernel_size': 5, 'dilation': 1},
    {'out_channels': 64, 'kernel_size': 7, 'dilation': 1},
    {'out_channels': 64, 'kernel_size': 3, 'dilation': 2},
    {'out_channels': 64, 'kernel_size': 5, 'dilation': 2},
    {'out_channels': 32, 'kernel_size': 15, 'dilation': 1},
]

LSTM_HIDDEN_SIZE = 128
LSTM_NUM_LAYERS = 2
LSTM_BIDIRECTIONAL = True
FC_LAYERS_SIZES = [512, 256, 128]

def create_fixed_vocab_45():
    """创建固定的45大小词汇表，匹配预训练模型"""
    chars = [
        'C', 'N', 'O', 'S', 'P', 'F', 'Cl', 'Br', 'I',
        'c', 'n', 'o', 's', 'p',
        '(', ')', '[', ']',
        '=', '#', '-',
        '+',
        '1', '2', '3', '4', '5', '6', '7', '8', '9', '0',
        '@', '@@',
        '%',
        '.',
        '*',
        '/', '\\',
        'H', 'B',
    ]
    unique_chars = sorted(list(set(chars)))
    additional_chars = ['Si', 'Se', 'As', 'Te', 'Al', 'Ga', 'Ge', 'Sn', 'Pb', 'Sb', 'Bi', 'X', 'Y', 'Z']
    for char in additional_chars:
        if char not in unique_chars:
            unique_chars.append(char)
            if len(unique_chars) >= 43:
                break
    unique_chars = unique_chars[:43]
    char_dict = {char: i + 1 for i, char in enumerate(unique_chars)}
    char_dict['&'] = 0
    char_dict['!'] = len(char_dict)
    print(f"Created fixed vocab with size: {len(char_dict)}")
    return char_dict

def set_seed(seed_value=42):
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed_value)
        torch.cuda.manual_seed_all(seed_value)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def clean_and_validate_smiles(smiles):
    """清洗和验证SMILES"""
    if not isinstance(smiles, str) or len(smiles) == 0:
        return None
    bad_patterns = ['[R]', '[R1]', '[R2]', '[R3]', '[R4]', '[R5]']
    for pattern in bad_patterns:
        if pattern in smiles:
            return None
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is not None:
            return Chem.MolToSmiles(mol, canonical=True)
        else:
            return None
    except:
        return None

# ==========================================
# 外部数据加载 & 绘图模块
# ==========================================
def load_and_merge_pi1m(train_df):
    """加载 Kaggle 上的 PI1M 数据集并合并到训练集"""
    base_path = "/kaggle/input/datasets/jsday96/pi1m-pseudolabels"
    if not os.path.exists(base_path):
        base_path = "/kaggle/input/pi1m-pseudolabels" # 备用路径
        if not os.path.exists(base_path):
            print("找不到 PI1M 数据，跳过合并。")
            return train_df

    pi1m_dfs = []
    for v in ['v2.1', 'v2.2', 'v3']:
        file_path = os.path.join(base_path, f"PI1M_50000_{v}.csv")
        if os.path.exists(file_path):
            pi1m_dfs.append(pd.read_csv(file_path))

    if not pi1m_dfs: return train_df

    print("正在合并 PI1M 伪标签...")
    pi1m_combined = pd.concat(pi1m_dfs, ignore_index=True)
    pi1m_combined['SMILES'] = pi1m_combined['SMILES'].apply(clean_and_validate_smiles)
    pi1m_combined = pi1m_combined.dropna(subset=['SMILES'])
    
    numeric_cols = pi1m_combined.select_dtypes(include=[np.number]).columns.tolist()
    pi1m_combined = pi1m_combined.groupby('SMILES')[numeric_cols].mean().reset_index()

    for target in TARGET_COLUMNS:
        actual_col = target if target in pi1m_combined.columns else target.lower()
        if actual_col in pi1m_combined.columns:
            df_target = pi1m_combined[['SMILES', actual_col]].rename(columns={actual_col: target})
            train_df = add_extra_data(train_df, df_target, target)
    return train_df

def plot_loss_curve(train_loss, val_loss, title, save_path):
    plt.figure(figsize=(8, 6))
    plt.plot(train_loss, label='Train Loss', linewidth=2)
    if val_loss: plt.plot(val_loss, label='Validation Loss', linewidth=2)
    plt.title(f'Figure 2: {title}')
    plt.xlabel('Epochs'); plt.ylabel('Loss')
    plt.legend(); plt.tight_layout()
    plt.savefig(save_path, dpi=300); plt.close()

def plot_parity(y_true, y_pred, title, save_path):
    plt.figure(figsize=(7, 7))
    plt.scatter(y_true, y_pred, alpha=0.6, color='#2ca02c')
    min_val, max_val = min(min(y_true), min(y_pred)), max(max(y_true), max(y_pred))
    plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Ideal')
    plt.title(f'Figure 3: Parity Plot - {title}')
    plt.xlabel('True'); plt.ylabel('Predicted')
    plt.legend(); plt.tight_layout()
    plt.savefig(save_path, dpi=300); plt.close()

def plot_residuals(y_true, y_pred, title, save_path):
    plt.figure(figsize=(8, 6))
    sns.histplot(np.array(y_true) - np.array(y_pred), kde=True, bins=30)
    plt.axvline(0, color='r', linestyle='--')
    plt.title(f'Figure 4: Error Distribution - {title}')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300); plt.close()

def plot_property_comparison(properties, scores, save_path):
    plt.figure(figsize=(9, 6))
    bars = plt.bar(properties, scores, color='#17becf')
    for bar in bars:
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{bar.get_height():.4f}', ha='center')
    plt.title('Figure 5: Prediction Performance (MAE)')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300); plt.close()

# ===============================
# 修复的主链/侧链特征提取
# ===============================

def get_backbone_sidechain_features(smiles: str) -> dict:
    """改进的主链/侧链特征提取 - 更好的错误处理"""
    feature_keys = [
        'bb_atom_count', 'sc_atom_count', 'sc_to_bb_atom_ratio',
        'bb_mw', 'sc_mw', 'sc_to_bb_mw_ratio',
        'bb_rotatable_bonds', 'sc_rotatable_bonds',
        'bb_aromatic_rings', 'sc_aromatic_rings',
        'bb_tpsa', 'sc_tpsa',
        'avg_sc_fragment_size', 'avg_sc_fragment_mw',
        'num_sc_fragments', "stddev_sc_fragment_size", "stddev_sc_fragment_mw"
    ]
    features = {key: 0.0 for key in feature_keys}
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None: return features
        star_atoms = [a for a in mol.GetAtoms() if a.GetAtomicNum() == 0]
        if len(star_atoms) != 2: return features
        if not star_atoms[0].GetNeighbors() or not star_atoms[1].GetNeighbors(): return features
        endpoint1 = star_atoms[0].GetNeighbors()[0].GetIdx()
        endpoint2 = star_atoms[1].GetNeighbors()[0].GetIdx()
        if endpoint1 == endpoint2:
            features['bb_atom_count'] = 1.0
            features['sc_atom_count'] = 0.0
            return features
        backbone_path = rdmolops.GetShortestPath(mol, endpoint1, endpoint2)
        if not backbone_path: return features
        backbone_atom_indices = set(backbone_path)
        all_real_atom_indices = {a.GetIdx() for a in mol.GetAtoms() if a.GetAtomicNum() != 0}
        sidechain_atom_indices = all_real_atom_indices - backbone_atom_indices
        features['bb_atom_count'] = float(len(backbone_atom_indices))
        features['sc_atom_count'] = float(len(sidechain_atom_indices))
        if features['bb_atom_count'] > 0:
            features['sc_to_bb_atom_ratio'] = features['sc_atom_count'] / features['bb_atom_count']
        try:
            bb_mw = sum(mol.GetAtomWithIdx(i).GetMass() for i in backbone_atom_indices)
            sc_mw = sum(mol.GetAtomWithIdx(i).GetMass() for i in sidechain_atom_indices)
            features['bb_mw'] = float(bb_mw) if not np.isnan(bb_mw) else 0.0
            features['sc_mw'] = float(sc_mw) if not np.isnan(sc_mw) else 0.0
            if bb_mw > 0:
                features['sc_to_bb_mw_ratio'] = float(sc_mw / bb_mw)
        except: pass
    except Exception as e: pass
    for key in features:
        if features[key] is None or np.isnan(features[key]) or np.isinf(features[key]):
            features[key] = 0.0
        else:
            features[key] = float(features[key])
    return features

def add_backbone_sidechain_features_to_df(df):
    """将主链和侧链特征计算并添加到DataFrame中"""
    features = df['SMILES'].apply(get_backbone_sidechain_features)
    feature_df = pd.json_normalize(features)
    feature_df = feature_df.fillna(0.0)
    for col in feature_df.columns:
        feature_df[col] = feature_df[col].replace([np.inf, -np.inf], 0.0)
    return pd.concat([df, feature_df], axis=1)

BACKBONE_SIDECHAIN_FEATURE_NAMES = [
    'bb_atom_count', 'sc_atom_count', 'sc_to_bb_atom_ratio',
    'bb_mw', 'sc_mw', 'sc_to_bb_mw_ratio',
    'bb_rotatable_bonds', 'sc_rotatable_bonds',
    'bb_aromatic_rings', 'sc_aromatic_rings',
    'bb_tpsa', 'sc_tpsa',
    'avg_sc_fragment_size', 'avg_sc_fragment_mw',
    'num_sc_fragments', "stddev_sc_fragment_size", "stddev_sc_fragment_mw"
]

# ===============================
# 数据清理函数
# ===============================

def clean_features_dataframe(df):
    """清理DataFrame中的NaN值"""
    for col in BACKBONE_SIDECHAIN_FEATURE_NAMES:
        if col in df.columns:
            df[col] = df[col].fillna(0.0)
            df[col] = df[col].replace([np.inf, -np.inf], 0.0)
    return df

# ===============================
# 保持原有的数据增强
# ===============================

class SMILESAugmenter:
    """SMILES数据增强类"""
    def __init__(self, augmentation_prob=0.3):
        self.augmentation_prob = augmentation_prob
    def random_smiles(self, smiles):
        try:
            mol = Chem.MolFromSmiles(smiles)
            if mol is None: return smiles
            new_mol = Chem.RenumberAtoms(mol, np.random.permutation(mol.GetNumAtoms()).tolist())
            new_smiles = Chem.MolToSmiles(new_mol, canonical=False)
            if Chem.MolFromSmiles(new_smiles) is not None: return new_smiles
            else: return smiles
        except: return smiles
    def enumerate_smiles(self, smiles):
        try:
            mol = Chem.MolFromSmiles(smiles)
            if mol is None: return smiles
            augmented_smiles = Chem.MolToSmiles(mol, canonical=False, doRandom=True)
            if Chem.MolFromSmiles(augmented_smiles) is not None: return augmented_smiles
            else: return smiles
        except: return smiles
    def augment_smiles(self, smiles):
        if random.random() > self.augmentation_prob: return smiles
        method = random.choice([self.random_smiles, self.enumerate_smiles])
        return method(smiles)

# ===============================
# 数据集类 
# ===============================

class SimpleMolFormerDataset(Dataset):
    def __init__(self, smiles_list, targets, backbone_features,
                 char_dict, max_len, is_training=True, augmenter=None):
        self.smiles_list = smiles_list
        self.targets = targets
        self.backbone_features = backbone_features
        self.char_dict = char_dict
        self.max_len = max_len
        self.pad_token_id = char_dict['&']
        self.unknown_token_id = char_dict['!']
        self.is_training = is_training
        self.augmenter = augmenter

    def __len__(self):
        return len(self.smiles_list)

    def __getitem__(self, idx):
        smiles = self.smiles_list[idx]
        target = self.targets[idx]
        backbone_feats = self.backbone_features[idx]
        
        backbone_feats = np.nan_to_num(backbone_feats, nan=0.0, posinf=1e6, neginf=-1e6)
        
        if self.is_training and self.augmenter is not None and DATA_AUGMENTATION:
            smiles = self.augmenter.augment_smiles(smiles)
        
        tokenized_smiles = [self.char_dict.get(char, self.unknown_token_id) for char in smiles]
        padding_len = self.max_len - len(tokenized_smiles)
        
        if padding_len > 0:
            padded_smiles = tokenized_smiles + [self.pad_token_id] * padding_len
        else:
            padded_smiles = tokenized_smiles[:self.max_len]
        
        return (torch.tensor(padded_smiles, dtype=torch.long),
                torch.tensor(backbone_feats, dtype=torch.float),
                torch.tensor(target, dtype=torch.float))

# ===============================
# 注意力机制
# ===============================

class SimpleAttention(nn.Module):
    """简化的注意力机制"""
    def __init__(self, hidden_size, dropout_rate=0.1):
        super(SimpleAttention, self).__init__()
        self.hidden_size = hidden_size
        self.attention_weights = nn.Linear(hidden_size, 1, bias=False)
        self.dropout = nn.Dropout(dropout_rate)
    def forward(self, lstm_output, mask=None):
        attention_scores = self.attention_weights(lstm_output).squeeze(-1)
        if mask is not None:
            attention_scores.masked_fill_(mask == 0, -1e9)
        attention_weights = torch.softmax(attention_scores, dim=1)
        attention_weights = self.dropout(attention_weights)
        weighted_output = torch.sum(lstm_output * attention_weights.unsqueeze(-1), dim=1)
        return weighted_output, attention_weights

# ===============================
# 模型 
# ===============================

class SimpleMolFormerCNNLSTM(nn.Module):
    """(已移除MolFormer) 修复了数值稳定性的CNN-LSTM模型"""
    def __init__(self, vocab_size, embedding_dim, cnn_configs, lstm_hidden_size,
                 lstm_num_layers, lstm_bidirectional, fc_layers_sizes,
                 num_backbone_features, dropout_rate=0.3):
        super(SimpleMolFormerCNNLSTM, self).__init__()
        
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.embedding_dropout = nn.Dropout(dropout_rate * 0.5)
        
        self.cnn_layers = nn.ModuleList()
        total_cnn_out_channels = 0
        for config in cnn_configs:
            self.cnn_layers.append(nn.Conv1d(
                embedding_dim, config['out_channels'], config['kernel_size'],
                dilation=config.get('dilation', 1), padding='same'
            ))
            total_cnn_out_channels += config['out_channels']
        
        self.global_max_pool = nn.AdaptiveMaxPool1d(1)
        self.global_avg_pool = nn.AdaptiveAvgPool1d(1)
        
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=lstm_hidden_size,
            num_layers=lstm_num_layers,
            batch_first=True,
            bidirectional=lstm_bidirectional,
            dropout=dropout_rate if lstm_num_layers > 1 else 0
        )
        lstm_output_size = lstm_hidden_size * (2 if lstm_bidirectional else 1)
        self.lstm_dropout = nn.Dropout(dropout_rate)
        
        self.attention = SimpleAttention(lstm_output_size, dropout_rate)
        
        backbone_processor_output_size = 64
        self.backbone_processor = nn.Sequential(
            nn.Linear(num_backbone_features, backbone_processor_output_size),
            nn.ReLU(),
            nn.Dropout(dropout_rate)
        )
        
        cnn_feature_size = total_cnn_out_channels * 2
        lstm_feature_size = lstm_output_size
        combined_input_size = (cnn_feature_size + lstm_feature_size +
                             backbone_processor_output_size)
        
        self.fc_layers = nn.ModuleList()
        self.fc_layer_norms = nn.ModuleList()
        self.fc_dropouts = nn.ModuleList()
        
        prev_size = combined_input_size
        for layer_size in fc_layers_sizes:
            self.fc_layers.append(nn.Linear(prev_size, layer_size))
            self.fc_layer_norms.append(nn.LayerNorm(layer_size))
            self.fc_dropouts.append(nn.Dropout(dropout_rate))
            prev_size = layer_size
        
        self.output_layer = nn.Linear(fc_layers_sizes[-1], 1)
        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None: nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                if m.bias is not None: nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.LSTM):
                for name, param in m.named_parameters():
                    if 'weight_ih' in name: nn.init.xavier_normal_(param)
                    elif 'weight_hh' in name: nn.init.orthogonal_(param)
                    elif 'bias' in name: nn.init.constant_(param, 0)
            elif isinstance(m, (nn.LayerNorm, nn.BatchNorm1d)):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def create_padding_mask(self, x, pad_token_id=0):
        return (x != pad_token_id).float()

    def forward(self, x, backbone_features):
        if torch.isnan(backbone_features).any() or torch.isinf(backbone_features).any():
            backbone_features = torch.nan_to_num(backbone_features, nan=0.0, posinf=1e6, neginf=-1e6)
        
        padding_mask = self.create_padding_mask(x)
        embedded = self.embedding_dropout(self.embedding(x))
        if torch.isnan(embedded).any():
            embedded = torch.nan_to_num(embedded, nan=0.0)
        
        cnn_input = embedded.transpose(1, 2)
        cnn_features = []
        for cnn_layer in self.cnn_layers:
            cnn_output = F.relu(cnn_layer(cnn_input))
            if torch.isnan(cnn_output).any(): cnn_output = torch.nan_to_num(cnn_output, nan=0.0)
            cnn_features.extend([self.global_max_pool(cnn_output).squeeze(-1), self.global_avg_pool(cnn_output).squeeze(-1)])
        cnn_combined = torch.cat(cnn_features, dim=1)
        
        lstm_out, _ = self.lstm(embedded)
        lstm_out = self.lstm_dropout(lstm_out)
        if torch.isnan(lstm_out).any():
            lstm_out = torch.nan_to_num(lstm_out, nan=0.0)
        
        attn_output, _ = self.attention(lstm_out, padding_mask)
        
        processed_backbone_features = self.backbone_processor(backbone_features)
        
        features_to_check = [cnn_combined, attn_output, processed_backbone_features]
        cleaned_features = []
        for feature in features_to_check:
            if torch.isnan(feature).any() or torch.isinf(feature).any():
                feature = torch.nan_to_num(feature, nan=0.0, posinf=1e6, neginf=-1e6)
            cleaned_features.append(feature)
        
        combined_features = torch.cat(cleaned_features, dim=1)
        
        x = combined_features
        for fc, ln, dropout in zip(self.fc_layers, self.fc_layer_norms, self.fc_dropouts):
            x = dropout(torch.relu(ln(fc(x))))
            if torch.isnan(x).any(): x = torch.nan_to_num(x, nan=0.0)
        
        output = self.output_layer(x).squeeze(1)
        if torch.isnan(output).any() or torch.isinf(output).any():
            output = torch.nan_to_num(output, nan=0.0, posinf=1e6, neginf=-1e6)
        return output

# ===============================
# 模型保存和加载函数
# ===============================

def save_model_and_metadata(model, target_col, fold, char_dict, config, model_dir='./models'):
    os.makedirs(model_dir, exist_ok=True)
    model_path = os.path.join(model_dir, f'model_{target_col}_fold_{fold}.pth')
    torch.save(model.state_dict(), model_path)
    metadata = {
        'target_col': target_col,
        'fold': fold,
        'char_dict': char_dict,
        'model_config': config,
        'vocab_size': len(char_dict),
        'backbone_features': BACKBONE_SIDECHAIN_FEATURE_NAMES,
    }
    metadata_path = os.path.join(model_dir, f'metadata_{target_col}_fold_{fold}.json')
    with open(metadata_path, 'w') as f:
        json.dump(metadata, f, indent=2)
    print(f"Saved model and metadata for {target_col} fold {fold}")
    return model_path, metadata_path

def load_model_and_metadata(target_col, fold, model_dir='./models'):
    metadata_path = os.path.join(model_dir, f'metadata_{target_col}_fold_{fold}.json')
    model_path = os.path.join(model_dir, f'model_{target_col}_fold_{fold}.pth')
    with open(metadata_path, 'r') as f:
        metadata = json.load(f)
    model = SimpleMolFormerCNNLSTM(
        vocab_size=metadata['vocab_size'],
        embedding_dim=SMILES_EMBEDDING_DIM,
        cnn_configs=CNN_CONFIGS,
        lstm_hidden_size=LSTM_HIDDEN_SIZE,
        lstm_num_layers=LSTM_NUM_LAYERS,
        lstm_bidirectional=LSTM_BIDIRECTIONAL,
        fc_layers_sizes=FC_LAYERS_SIZES,
        num_backbone_features=len(metadata['backbone_features']),
        dropout_rate=metadata['model_config']['dropout_rate']
    )
    model.load_state_dict(torch.load(model_path))
    return model, metadata

# ===============================
# 修复的训练和评估函数
# ===============================

def get_model_config(target_col):
    if target_col in ['FFV', 'Tc']:
        return {'dropout_rate': 0.05, 'weight_decay': 1e-3, 'learning_rate': LEARNING_RATE * 0.5, 'patience': 20}
    else:
        return {'dropout_rate': DROPOUT_RATE, 'weight_decay': 1e-4, 'learning_rate': LEARNING_RATE, 'patience': 15}

def scale_target(values, target_col, inverse=False):
    scale_factor = TARGET_SCALING.get(target_col, 1)
    return values / scale_factor if inverse else values * scale_factor

def train_epoch(model, train_loader, optimizer, criterion, device, accumulation_steps=1):
    model.train()
    total_loss, num_batches = 0, 0
    optimizer.zero_grad()
    for batch_idx, (smiles, backbone_features, targets) in enumerate(train_loader):
        smiles, backbone_features, targets = (smiles.to(device), backbone_features.to(device), targets.to(device))
        predictions = model(smiles, backbone_features)
        loss = criterion(predictions, targets)
        if torch.isnan(loss) or torch.isinf(loss): continue
        loss = loss / accumulation_steps
        loss.backward()
        if (batch_idx + 1) % accumulation_steps == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()
        total_loss += loss.item() * accumulation_steps
        num_batches += 1
    if num_batches % accumulation_steps != 0:
        optimizer.step()
        optimizer.zero_grad()
    return total_loss / max(num_batches, 1)

def validate_epoch(model, val_loader, criterion, device, target_col):
    model.eval()
    total_loss, num_batches = 0, 0
    all_predictions, all_targets = [], []
    with torch.no_grad():
        for smiles, backbone_features, targets in val_loader:
            smiles, backbone_features, targets = (smiles.to(device), backbone_features.to(device), targets.to(device))
            if torch.isnan(backbone_features).any():
                backbone_features = torch.nan_to_num(backbone_features, nan=0.0)
            predictions = model(smiles, backbone_features)
            if torch.isnan(predictions).any() or torch.isinf(predictions).any():
                predictions = torch.nan_to_num(predictions, nan=0.0, posinf=1e6, neginf=-1e6)
            loss = criterion(predictions, targets)
            if torch.isnan(loss) or torch.isinf(loss): continue
            total_loss += loss.item()
            num_batches += 1
            all_predictions.extend(predictions.cpu().numpy())
            all_targets.extend(targets.cpu().numpy())
    if num_batches == 0: return float('inf'), float('inf')
    all_predictions, all_targets = np.array(all_predictions), np.array(all_targets)
    all_predictions = np.nan_to_num(all_predictions, nan=0.0, posinf=1e6, neginf=-1e6)
    all_targets = np.nan_to_num(all_targets, nan=0.0, posinf=1e6, neginf=-1e6)
    try:
        original_predictions = scale_target(all_predictions, target_col, inverse=True)
        original_targets = scale_target(all_targets, target_col, inverse=True)
        valid_mask = ~(np.isnan(original_predictions) | np.isnan(original_targets))
        if valid_mask.sum() > 0:
            mae = mean_absolute_error(original_targets[valid_mask], original_predictions[valid_mask])
        else: mae = float('inf')
    except Exception as e: mae = float('inf')
    return total_loss / num_batches, mae

def train_model(model, train_loader, val_loader, optimizer, scheduler, criterion, epochs, device, target_col, patience=15):
    model.to(device)
    best_val_loss, best_mae, patience_counter = np.inf, np.inf, 0
    best_model_state = None
    for epoch in range(epochs):
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_mae = validate_epoch(model, val_loader, criterion, device, target_col)
        if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau): scheduler.step(val_loss)
        else: scheduler.step()
        print(f"Epoch {epoch+1}/{epochs} -> Train Loss: {train_loss:.6f}, Val Loss: {val_loss:.6f}, Val MAE: {val_mae:.6f}")
        if val_loss < best_val_loss:
            best_val_loss, best_mae = val_loss, val_mae
            best_model_state = model.state_dict().copy()
            patience_counter = 0
            print(f"  -> Better model found! Val Loss: {best_val_loss:.6f}, Val MAE: {val_mae:.6f}")
        else:
            patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping triggered after {epoch+1} epochs")
            break
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print(f"\nLoaded best model with Val Loss: {best_val_loss:.6f}, Final MAE: {best_mae:.6f}")
    return model, best_mae

def add_extra_data(df_train, df_extra, target):
    n_samples_before = len(df_train[df_train[target].notnull()])
    df_extra['SMILES'] = df_extra['SMILES'].apply(clean_and_validate_smiles)
    df_extra = df_extra.dropna(subset=['SMILES', target]).groupby('SMILES', as_index=False)[target].mean()
    cross_smiles = set(df_extra['SMILES']) & set(df_train['SMILES'])
    unique_smiles_extra = set(df_extra['SMILES']) - set(df_train['SMILES'])
    for smile in cross_smiles:
        if pd.isnull(df_train.loc[df_train['SMILES']==smile, target]).any():
            impute_value = df_extra[df_extra['SMILES']==smile][target].values[0]
            df_train.loc[df_train['SMILES']==smile, target] = impute_value
    df_train = pd.concat([df_train, df_extra[df_extra['SMILES'].isin(unique_smiles_extra)]], axis=0, ignore_index=True)
    n_samples_after = len(df_train[df_train[target].notnull()])
    print(f'For target "{target}": {n_samples_after - n_samples_before} new samples were added.')
    return df_train

# ===============================
# 主函数 
# ===============================

def main():
    set_seed(SEED)
    print(f"Using device: {DEVICE}")
    print(f"MolFormer available: {MOLFORMER_AVAILABLE}")
    print(f"Data Augmentation: {'Enabled' if DATA_AUGMENTATION else 'Disabled'}")
    
    try:
        train_df = pd.read_csv(BASE_PATH + 'train.csv')
        test_df = pd.read_csv(BASE_PATH + 'test.csv')
    except FileNotFoundError:
        print(f"Error: Data files not found at {BASE_PATH}")
        return
    
    train_df['SMILES'] = train_df['SMILES'].apply(clean_and_validate_smiles)
    test_df['SMILES'] = test_df['SMILES'].apply(clean_and_validate_smiles)
    train_df = train_df[train_df['SMILES'].notnull()].reset_index(drop=True)
    test_df = test_df[test_df['SMILES'].notnull()].reset_index(drop=True)
    
    print("Calculating backbone/sidechain features...")
    train_df = add_backbone_sidechain_features_to_df(train_df)
    test_df = add_backbone_sidechain_features_to_df(test_df)
    train_df = clean_features_dataframe(train_df)
    test_df = clean_features_dataframe(test_df)
    
    try:
        extra_paths = {
            'tc': BASE_PATH + 'train_supplement/dataset1.csv',
            'tg': BASE_PATH + 'train_supplement/dataset3.csv',
            'ffv': BASE_PATH + 'train_supplement/dataset4.csv'
        }
        data_tc = pd.read_csv(extra_paths['tc']).rename(columns={'TC_mean': 'Tc'})
        train_df = add_extra_data(train_df, data_tc, 'Tc')
        data_tg = pd.read_csv(extra_paths['tg'], usecols=['SMILES', 'Tg'])
        train_df = add_extra_data(train_df, data_tg, 'Tg')
        data_ffv = pd.read_csv(extra_paths['ffv'], usecols=['SMILES', 'FFV'])
        train_df = add_extra_data(train_df, data_ffv, 'FFV')
    except Exception as e:
        print(f"Warning: Could not load extra data: {e}")
    
    # 引入 PI1M 伪标签
    train_df = load_and_merge_pi1m(train_df)
    
    smiles_char_dict = create_fixed_vocab_45()
    vocab_size = len(smiles_char_dict)
    max_smiles_len = train_df['SMILES'].astype(str).str.len().max() + 20
    print(f"Fixed Vocabulary Size: {vocab_size}")
    print(f"Max SMILES Length: {max_smiles_len}")
    
    augmenter = SMILESAugmenter(augmentation_prob=AUGMENTATION_PROBABILITY)
    TRAINING_RESULTS = {}
    os.makedirs('./models', exist_ok=True)
    with open('./models/char_dict.json', 'w') as f:
        json.dump(smiles_char_dict, f, indent=2)
    
    for target_col in TARGET_COLUMNS:
        print(f"\n{'='*50}")
        print(f"Training Simple CNN-LSTM (No MolFormer) for: {target_col}")
        print(f"{'='*50}")
        
        df_filtered = train_df.dropna(subset=[target_col]).copy().reset_index(drop=True)
        if len(df_filtered) < 20:
            print(f"Not enough data for '{target_col}'. Skipping...")
            continue
        
        print(f"{target_col} statistics:")
        print(f"Range: {df_filtered[target_col].min():.6f} - {df_filtered[target_col].max():.6f}")
        print(f"Mean: {df_filtered[target_col].mean():.6f}, Std: {df_filtered[target_col].std():.6f}")
        print(f"Unique values: {df_filtered[target_col].nunique()}")
        
        df_filtered[target_col] = scale_target(df_filtered[target_col].values, target_col)
        
        model_config = get_model_config(target_col)
        print(f"Using config: LR={model_config['learning_rate']:.6f}, "
              f"Dropout={model_config['dropout_rate']:.3f}, "
              f"Patience={model_config['patience']}")
        
        try:
            y_bins = pd.qcut(df_filtered[target_col], q=10, labels=False, duplicates='drop')
            stratify_param = y_bins
        except ValueError:
            stratify_param = None
        
        kfold = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
        fold_models, fold_maes = [], []
        
        # 修复2：提前初始化 OOF 列表
        oof_true_list = []
        oof_pred_list = []
        
        for fold, (train_idx, val_idx) in enumerate(kfold.split(df_filtered['SMILES'], stratify_param)):
            print(f"\nTraining Fold {fold+1}/{N_FOLDS}")
            
            X_train_smiles = df_filtered.iloc[train_idx]['SMILES'].values
            y_train = df_filtered.iloc[train_idx][target_col].values
            X_train_backbone = df_filtered.iloc[train_idx][BACKBONE_SIDECHAIN_FEATURE_NAMES].values
            
            X_val_smiles = df_filtered.iloc[val_idx]['SMILES'].values
            y_val = df_filtered.iloc[val_idx][target_col].values
            X_val_backbone = df_filtered.iloc[val_idx][BACKBONE_SIDECHAIN_FEATURE_NAMES].values
            
            train_dataset = SimpleMolFormerDataset(
                X_train_smiles, y_train, X_train_backbone,
                smiles_char_dict, max_smiles_len, is_training=True, augmenter=augmenter
            )
            val_dataset = SimpleMolFormerDataset(
                X_val_smiles, y_val, X_val_backbone,
                smiles_char_dict, max_smiles_len, is_training=False, augmenter=None
            )
            
            train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
            val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
            
            model = SimpleMolFormerCNNLSTM(
                vocab_size=vocab_size,
                embedding_dim=SMILES_EMBEDDING_DIM,
                cnn_configs=CNN_CONFIGS,
                lstm_hidden_size=LSTM_HIDDEN_SIZE,
                lstm_num_layers=LSTM_NUM_LAYERS,
                lstm_bidirectional=LSTM_BIDIRECTIONAL,
                fc_layers_sizes=FC_LAYERS_SIZES,
                num_backbone_features=len(BACKBONE_SIDECHAIN_FEATURE_NAMES),
                dropout_rate=model_config['dropout_rate']
            )
            
            optimizer = optim.AdamW(model.parameters(), lr=model_config['learning_rate'], weight_decay=model_config['weight_decay'])
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=8, verbose=False)
            criterion = nn.MSELoss()
            
            model, fold_mae = train_model(model, train_loader, val_loader, optimizer, scheduler,
                                        criterion, EPOCHS, DEVICE, target_col, patience=model_config['patience'])
            model.eval()
            fold_true = []
            fold_pred = []
            with torch.no_grad():
                # 修复1：删除了多余的 molformer_features 解包
                for smiles, backbone_features, targets in val_loader:
                    smiles = smiles.to(DEVICE)
                    backbone_features = backbone_features.to(DEVICE)
                    
                    preds = model(smiles, backbone_features) 
                    
                    if torch.isnan(preds).any() or torch.isinf(preds).any():
                        preds = torch.nan_to_num(preds, nan=0.0, posinf=1e6, neginf=-1e6)
                        
                    fold_true.extend(targets.cpu().numpy())
                    fold_pred.extend(preds.cpu().numpy())
            
            # 逆缩放还原到真实尺度，并追加到全局列表中
            oof_true_list.extend(scale_target(np.array(fold_true), target_col, inverse=True))
            oof_pred_list.extend(scale_target(np.array(fold_pred), target_col, inverse=True))
            # ==========================================
            
            save_model_and_metadata(model, target_col, fold, smiles_char_dict, model_config)
            fold_models.append(model.state_dict().copy())
            fold_maes.append(fold_mae)
        
        avg_mae = np.mean(fold_maes)
        plot_parity(oof_true_list, oof_pred_list, title=target_col, save_path=f"./plots/fig3_parity_{target_col}.png")
        plot_residuals(oof_true_list, oof_pred_list, title=target_col, save_path=f"./plots/fig4_residuals_{target_col}.png")
        TRAINING_RESULTS[target_col] = {'avg_mae': avg_mae, 'fold_maes': fold_maes, 'data_samples': len(df_filtered), 'models': fold_models}
        print(f"\n{target_col} K-Fold Average MAE: {avg_mae:.6f}")
    
    print("\n" + "="*50 + "\nPredicting test data...\n" + "="*50)
    submission_df = pd.DataFrame({'id': test_df['id']})
    
    for target_col in TARGET_COLUMNS:
        if target_col not in TRAINING_RESULTS:
            submission_df[target_col] = 0.0
            continue
        
        print(f"\nPredicting for: {target_col}")
        fold_predictions = []
        model_config = get_model_config(target_col)
        
        for fold, model_state in enumerate(TRAINING_RESULTS[target_col]['models']):
            model = SimpleMolFormerCNNLSTM(
                vocab_size=vocab_size, embedding_dim=SMILES_EMBEDDING_DIM, cnn_configs=CNN_CONFIGS,
                lstm_hidden_size=LSTM_HIDDEN_SIZE, lstm_num_layers=LSTM_NUM_LAYERS, lstm_bidirectional=LSTM_BIDIRECTIONAL,
                fc_layers_sizes=FC_LAYERS_SIZES, num_backbone_features=len(BACKBONE_SIDECHAIN_FEATURE_NAMES),
                dropout_rate=model_config['dropout_rate']
            )
            model.load_state_dict(model_state)
            model.to(DEVICE)
            model.eval()
            
            test_dataset = SimpleMolFormerDataset(
                test_df['SMILES'].values, np.zeros(len(test_df)),
                test_df[BACKBONE_SIDECHAIN_FEATURE_NAMES].values,
                smiles_char_dict, max_smiles_len, is_training=False, augmenter=None
            )
            test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
            
            predictions = []
            with torch.no_grad():
                # 测试集推理同样需要保持只有 3 个元素的解包
                for smiles, backbone_features, _ in tqdm(test_loader, desc=f"Fold {fold+1}"):
                    smiles, backbone_features = smiles.to(DEVICE), backbone_features.to(DEVICE)
                    pred = model(smiles, backbone_features)
                    if torch.isnan(pred).any() or torch.isinf(pred).any():
                        pred = torch.nan_to_num(pred, nan=0.0, posinf=1e6, neginf=-1e6)
                    predictions.extend(pred.cpu().numpy())
            
            predictions = scale_target(np.array(predictions), target_col, inverse=True)
            fold_predictions.append(predictions)
        
        final_predictions = np.nan_to_num(np.mean(fold_predictions, axis=0), nan=0.0, posinf=1e6, neginf=-1e6)
        submission_df[target_col] = final_predictions
        
    # 画 Figure 5 对比图
    if len(TRAINING_RESULTS) > 0:
        targets_list = list(TRAINING_RESULTS.keys())
        maes_list = [TRAINING_RESULTS[t]['avg_mae'] for t in targets_list]
        plot_property_comparison(targets_list, maes_list, save_path="./plots/fig5_property_comparison.png")
    
    submission_df.to_csv('submission.csv', index=False)
    print(f'\nSubmission saved!')
    print("Sample predictions:")
    print(submission_df.head())
    
    with open('./models/training_results.json', 'w') as f:
        training_results_serializable = {}
        for target, results in TRAINING_RESULTS.items():
            training_results_serializable[target] = {
                'avg_mae': float(results['avg_mae']),
                'fold_maes': [float(mae) for mae in results['fold_maes']],
                'data_samples': results['data_samples']
            }
        json.dump(training_results_serializable, f, indent=2)
    
    print("\n" + "="*70 + "\n" + "                    TRAINING SUMMARY" + "\n" + "="*70)
    total_mae, valid_targets = 0, 0
    print(f"{'Target':<10} {'Data Samples':<12} {'Validation MAE':<15} {'Details':<25}")
    print("-" * 70)
    for target, results in TRAINING_RESULTS.items():
        data_samples, avg_mae, fold_maes = results['data_samples'], results['avg_mae'], results['fold_maes']
        mae_std = np.std(fold_maes)
        details = f"±{mae_std:.4f} (K-Fold)"
        total_mae += avg_mae
        valid_targets += 1
        print(f"{target:<10} {data_samples:<12} {avg_mae:<15.6f} {details:<25}")
    print("-" * 70)
    if valid_targets > 0:
        overall_avg_mae = total_mae / valid_targets
        print(f"{'OVERALL':<10} {'':<12} {overall_avg_mae:<15.6f} {'Average MAE':<25}")
    print(f"\nModel Configuration:")
    print(f"- Architecture: Fixed CNN-LSTM (No MolFormer features)")
    print(f"- Fixed Vocabulary Size: {vocab_size}")
    print(f"- MolFormer Available: {MOLFORMER_AVAILABLE}")
    print(f"- Data Augmentation: {'Enabled' if DATA_AUGMENTATION else 'Disabled'}")
    print(f"- Device: {DEVICE}")
    print(f"- Models saved to: ./models/")
    print("="*70)

if __name__ == "__main__":
    main()